# 02 - Train model A and model B

Both models are the same U-Net, trained in the same way. Only the split differs:

- **Model A**: validation frames drawn at random from the 14 non-test videos. Its validation frames come from the same videos, often from the same seconds, as its training frames.
- **Model B**: 11 videos for training, 3 other videos for validation.

Each model keeps the epoch with the best validation mean Dice. That number is what a report would usually show; H2 asks how far it is from the score on new videos. The test videos are not loaded here.

Kaggle: Accelerator GPU T4, Internet on (for the ImageNet weights), the dataset `newslab/cholecseg8k` and the output of notebook 01 as inputs.
Output: `out/model_A.pt`, `out/model_B.pt`, `out/train_log_A.csv`, `out/train_log_B.csv`, `out/training.png`, plus the split files from 01, zipped as `results_02.zip`.

In [ ]:
import glob, json, math, os, re, shutil, time
from pathlib import Path
from multiprocessing import Pool
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

KAGGLE = Path('/kaggle').exists()
WORK = Path('/kaggle/working') if KAGGLE else Path('.')
DATA = Path(os.environ.get('DATA_ROOT', '/kaggle/input'))     # where the CholecSeg8k frames are
INPUT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))   # where the output of the previous notebook is
OUT = WORK / 'out'
OUT.mkdir(exist_ok=True)
H, W = 256, 448   # frames are 480 x 854; resized to a size the network can halve five times

CLASSES = ['black background', 'abdominal wall', 'liver', 'gastrointestinal tract', 'fat', 'grasper',
           'connective tissue', 'blood', 'cystic duct', 'L-hook electrocautery', 'gallbladder',
           'hepatic vein', 'liver ligament']
N_CLASSES = len(CLASSES)
IGNORE = 255      # mask pixels with a value that belongs to no class


def find_input(name):
    """Find a file from the previous notebook's output under INPUT."""
    hits = sorted(glob.glob(str(INPUT / '**' / name), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{name} not found under {INPUT}: add the output of the previous notebook as input')
    return Path(hits[0])

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PRETRAINED = os.environ.get('PRETRAINED', '1') == '1'   # ImageNet weights for the encoder (needs Internet on)
torch.manual_seed(0); np.random.seed(0)
print('device:', DEVICE)

split = pd.read_csv(find_input('split.csv'))
mapping = {int(k): v for k, v in json.loads(find_input('mapping.json').read_text()).items()}
classes_info = json.loads(find_input('classes.json').read_text())
LUT = np.full(256, IGNORE, np.uint8)            # mask value -> class id
for value, cls in mapping.items():
    LUT[value] = cls
print(split.groupby('split_B').video.nunique().to_dict(), 'videos per part (video split)')

## Frames

Only the 14 non-test videos are read. All frames go into memory once, resized to 256 x 448 (about 2.5 GB).

In [ ]:
def load_pair(path):
    """One frame and its class mask, both resized to H x W."""
    img = Image.open(path).convert('RGB').resize((W, H), Image.BILINEAR)
    m = np.array(Image.open(path.replace('_endo.png', '_endo_watershed_mask.png')))
    if m.ndim == 3:
        m = m[..., 0]                               # the three channels hold the same value
    m = Image.fromarray(LUT[m]).resize((W, H), Image.NEAREST)
    return np.asarray(img, np.uint8), np.asarray(m, np.uint8)


def load_frames(rows):
    """Read the frames of a table (one row per frame) into two arrays, in table order."""
    t0 = time.time()
    with Pool(os.cpu_count()) as pool:
        pairs = pool.map(load_pair, rows.path.tolist(), chunksize=16)
    imgs = np.stack([p[0] for p in pairs])
    masks = np.stack([p[1] for p in pairs])
    print(f'{len(rows)} frames loaded in {time.time() - t0:.0f} s, {imgs.nbytes / 1e9:.1f} GB')
    return imgs, masks

In [ ]:
rows = split[split.split_B != 'test'].reset_index(drop=True)
IMGS, MASKS = load_frames(rows)
idx_A = {s: np.flatnonzero(rows.split_A.values == s) for s in ['train', 'val']}
idx_B = {s: np.flatnonzero(rows.split_B.values == s) for s in ['train', 'val']}
print({k: len(v) for k, v in idx_A.items()}, 'frames, model A')
print({k: len(v) for k, v in idx_B.items()}, 'frames, model B')

## The model

A U-Net: the encoder makes the picture smaller step by step and learns what is in it; the decoder makes it big again and decides the class of every pixel. Skip connections pass the details of each encoder step to the decoder step of the same size.

Here the encoder is a ResNet-34 that was trained on ImageNet photos. With only 11 training videos this works much better than an encoder that starts from zero. The cell after the model checks the output shape.

In [ ]:
class DoubleConv(nn.Module):
    """Conv 3x3 -> BatchNorm -> ReLU, two times. Height and width stay the same."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    """Double the size, join the encoder output of the same size (skip connection), DoubleConv."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class UNetResNet34(nn.Module):
    """U-Net whose encoder is a ResNet-34 trained on ImageNet; the decoder is trained from scratch."""
    def __init__(self, n_classes=13, pretrained=True):
        super().__init__()
        r = torchvision.models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)   # 1/2 size, 64 channels
        self.pool = r.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.up4 = Up(512, 256, 256)                        # 1/32 -> 1/16
        self.up3 = Up(256, 128, 128)                        # 1/16 -> 1/8
        self.up2 = Up(128, 64, 64)                          # 1/8 -> 1/4
        self.up1 = Up(64, 64, 64)                           # 1/4 -> 1/2
        self.head = nn.Sequential(DoubleConv(64, 32), nn.Conv2d(32, n_classes, 1))

    def forward(self, x):
        s0 = self.stem(x)                     # 1/2
        s1 = self.layer1(self.pool(s0))       # 1/4, 64
        s2 = self.layer2(s1)                  # 1/8, 128
        s3 = self.layer3(s2)                  # 1/16, 256
        s4 = self.layer4(s3)                  # 1/32, 512
        x = self.up1(self.up2(self.up3(self.up4(s4, s3), s2), s1), s0)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.head(x)                   # 13 logits per pixel


MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_input(x_uint8):
    """(B, H, W, 3) uint8 -> (B, 3, H, W) float, normalised like the ImageNet training data."""
    x = x_uint8.to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
    return (x - MEAN) / STD

In [ ]:
m = UNetResNet34(N_CLASSES, pretrained=False).to(DEVICE)
with torch.no_grad():
    out = m(torch.randn(2, 3, H, W, device=DEVICE))
assert out.shape == (2, N_CLASSES, H, W), out.shape
print('ok, parameters:', round(sum(p.numel() for p in m.parameters()) / 1e6, 1), 'million')
del m

In [ ]:
@torch.no_grad()
def evaluate(model, imgs, masks, batch=32):
    """Run the model on frames and score it.

    Returns the confusion matrix over all frames (rows: true class, columns: predicted class) and
    one row per frame: mean Dice over the classes in that frame, and two uncertainty numbers.
    Pixels whose mask value belongs to no class are left out.
    """
    model.eval()
    total = torch.zeros(N_CLASSES * N_CLASSES, dtype=torch.long, device=DEVICE)
    rows = []
    for k in range(0, len(imgs), batch):
        x = to_input(torch.from_numpy(imgs[k:k + batch]))
        y = torch.from_numpy(masks[k:k + batch]).to(DEVICE).long()
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
            logits = model(x)
        prob = logits.float().softmax(1)
        pred = prob.argmax(1)
        entropy = -(prob * prob.clamp_min(1e-8).log()).sum(1)        # per pixel, 0 = certain
        for i in range(len(y)):
            valid = y[i] != IGNORE
            cm = torch.bincount(y[i][valid] * N_CLASSES + pred[i][valid], minlength=N_CLASSES ** 2)
            total += cm
            cm = cm.view(N_CLASSES, N_CLASSES).cpu().numpy()
            inter, true_px, pred_px = np.diag(cm), cm.sum(1), cm.sum(0)
            seen = (true_px + pred_px)[1:] > 0                         # classes 1-12 in truth or prediction
            dice = (2 * inter / np.maximum(true_px + pred_px, 1))[1:][seen]
            fg = pred[i] != 0                                          # everything except the black border
            rows.append(dict(frame_dice=float(dice.mean()) if len(dice) else np.nan,
                             entropy_fg=float(entropy[i][fg].mean()) if fg.any() else float(entropy[i].mean()),
                             entropy_all=float(entropy[i].mean())))
    return total.view(N_CLASSES, N_CLASSES).cpu().numpy(), pd.DataFrame(rows)


def dice_per_class(cm):
    """Dice per class over all frames; NaN for a class that is not in the ground truth."""
    inter, true_px, pred_px = np.diag(cm), cm.sum(1), cm.sum(0)
    return np.where(true_px > 0, 2 * inter / np.maximum(true_px + pred_px, 1), np.nan)


def mean_dice(cm):
    """Mean Dice over classes 1-12 that are in the ground truth (class 0 is the black border)."""
    return float(np.nanmean(dice_per_class(cm)[1:]))

## Training

- Loss: cross-entropy per pixel **plus** a Dice loss. Cross-entropy counts every pixel the same, so the large classes (liver, abdominal wall) dominate. The Dice loss gives every class in the batch the same weight, which helps small classes such as the grasper.
- Augmentation: random left-right flip, small changes of brightness and contrast.
- AdamW, learning rate up to 3e-4 with a short warm-up and cosine decay, batch 16, 15 epochs, mixed precision on the GPU.
- After every epoch: mean Dice on the validation frames; the best epoch is saved.

In [ ]:
EPOCHS = int(os.environ.get('EPOCHS', 15))
BATCH = int(os.environ.get('BATCH', 16))


def dice_loss(logits, target):
    valid = (target != IGNORE).unsqueeze(1)
    t = target.masked_fill(target == IGNORE, 0)
    prob = logits.float().softmax(1) * valid
    onehot = F.one_hot(t, N_CLASSES).permute(0, 3, 1, 2).float() * valid
    inter = (prob * onehot).sum((0, 2, 3))
    denom = prob.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    present = onehot.sum((0, 2, 3)) > 0
    return 1 - ((2 * inter + 1) / (denom + 1))[present].mean()


def augment(x, y):
    """x: (B, 3, H, W) normalised, y: (B, H, W). Flip and brightness/contrast, per frame."""
    flip = torch.rand(len(x), device=DEVICE) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(-1), x)
    y = torch.where(flip[:, None, None], y.flip(-1), y)
    contrast = torch.empty(len(x), 1, 1, 1, device=DEVICE).uniform_(0.8, 1.2)
    bright = torch.empty(len(x), 1, 1, 1, device=DEVICE).uniform_(-0.4, 0.4)
    return x * contrast + bright, y


def train(name, train_idx, val_idx):
    torch.manual_seed(0); np.random.seed(0)
    model = UNetResNet34(N_CLASSES, pretrained=PRETRAINED).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    steps = EPOCHS * (len(train_idx) // BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=max(steps, 1), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')
    log, best = [], -1.0
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); losses = []
        order = np.random.permutation(train_idx)
        for k in range(0, len(order) - BATCH + 1, BATCH):
            b = np.sort(order[k:k + BATCH])
            x = to_input(torch.from_numpy(IMGS[b]))
            y = torch.from_numpy(MASKS[b]).to(DEVICE).long()
            x, y = augment(x, y)
            with torch.autocast('cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
                logits = model(x)
            loss = F.cross_entropy(logits.float(), y, ignore_index=IGNORE) + dice_loss(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        cm, _ = evaluate(model, IMGS[val_idx], MASKS[val_idx])
        val_dice = mean_dice(cm)
        log.append(dict(epoch=epoch, train_loss=float(np.mean(losses)), val_mean_dice=val_dice, seconds=time.time() - t0))
        if val_dice > best:
            best = val_dice
            torch.save(model.state_dict(), OUT / f'model_{name}.pt')
        print(f'[{name}] epoch {epoch:2d}  loss {np.mean(losses):.3f}  val mean Dice {val_dice:.3f}  {time.time() - t0:.0f} s')
    log = pd.DataFrame(log)
    log.to_csv(OUT / f'train_log_{name}.csv', index=False)
    print(f'[{name}] best epoch {int(log.loc[log.val_mean_dice.idxmax(), "epoch"])}, val mean Dice {best:.3f}')
    return log

In [ ]:
log_A = train('A', idx_A['train'], idx_A['val'])

In [ ]:
log_B = train('B', idx_B['train'], idx_B['val'])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, log in [('A (random frames)', log_A), ('B (by video)', log_B)]:
    ax[0].plot(log.epoch, log.train_loss, label=name)
    ax[1].plot(log.epoch, log.val_mean_dice, label=name)
ax[0].set_ylabel('training loss'); ax[1].set_ylabel('validation mean Dice')
for a in ax:
    a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.savefig(OUT / 'training.png', dpi=110); plt.show()

for name in ['split.csv', 'mapping.json', 'classes.json', 'class_pixels.csv']:     # passed on to notebooks 03 and 04
    shutil.copy(find_input(name), OUT / name)
shutil.make_archive(str(WORK / 'results_02'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))